# IMS vs OL/DA Daily SCF Validation (M36 EASE Grid)

This notebook compares **regridded IMS daily categories on M36 EASE** against GEOSldas OL/DA daily model output
for a configurable year range.

It computes contingency-table skill metrics from binary snow/no-snow (threshold = 0.5):

- Accuracy
- Hit rate
- Miss rate
- False alarm ratio
- Correct rejection rate

It follows the SNOTEL notebook style with an intermediate cache so you can rerun statistics without re-extracting all daily files.


## Workflow

1. Configure year range, IMS regridded file path template, and OL/DA run roots.
2. Build deterministic representative tile mapping (one tile per M36 cell) from tilecoord.
3. For each day:
   - Read IMS regridded `ims_category` on M36.
   - Convert IMS category to binary snow/no-snow.
   - Read model SCF variable from daily cat file for OL and DA.
   - Compute daily A/B/C/D contingency counts and daily scores.
4. Save daily intermediate cache (`parquet` + `csv`).
5. Aggregate counts and recompute summary metrics by period/season/year.


In [ ]:
# -------------------------
# Imports + configuration
# -------------------------
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

SEASON_ORDER = ["DJF", "MAM", "JJA", "SON"]

EXPERIMENTS = {
    "OL": {
        "exp_name": "LS_OLv8_M36",
        "run_root": Path("/discover/nobackup/projects/land_da/M21C_land_sweeper/LS_OLv8_M36_v2/LS_OLv8_M36"),
    },
    "DA": {
        "exp_name": "LS_DAv8_M36",
        "run_root": Path("/discover/nobackup/projects/land_da/M21C_land_sweeper/LS_DAv8_M36_v3/LS_DAv8_M36"),
    },
}
DOMAIN = "SMAP_EASEv2_M36_GLOBAL"

YEAR_START = 2000
YEAR_END = 2023

# IMPORTANT: this notebook expects IMS already regridded to M36 EASE.
IMS_REGRID_DIR = Path("/gpfsm/dnb06/projects/p163/IMS")
IMS_REGRID_TEMPLATE = "ims_snowcover_24km_{year}_on_m36_nearest.nc4"
IMS_VAR = "ims_category"

# Candidate model snow-cover fraction variable names in daily cat files.
MODEL_SCF_VAR_CANDIDATES = ("FRLANDSNO", "FRSNO", "SNCOVFR", "SNOWCOVERFR", "SCF")
SCF_THRESHOLD = 0.5

# IMS category-to-binary mapping controls.
# For many IMS categorical products, code 4 = snow-covered land.
AUTO_INFER_IMS_CODES = True
IMS_SNOW_CODES = {4}
IMS_NO_SNOW_CODES = {0, 1, 2, 3}
IMS_FILL_VALUES = {-32768}

# Fair OL vs DA comparison controls.
# These enforce same collocated samples for OL and DA per day.
COMPARE_EXP_OL = "OL"
COMPARE_EXP_DA = "DA"

# Bootstrap controls (paired day-block bootstrap).
N_BOOTSTRAP = 2000
BOOTSTRAP_SEED = 42
CI_LOW = 2.5
CI_HIGH = 97.5

USE_DAILY_CACHE = True
WRITE_DAILY_CACHE = True

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "common/python/io/read_GEOSldas.py").exists():
    for parent in REPO_ROOT.parents:
        if (parent / "common/python/io/read_GEOSldas.py").exists():
            REPO_ROOT = parent
            break

if not (REPO_ROOT / "common/python/io/read_GEOSldas.py").exists():
    raise FileNotFoundError("Could not find common/python/io/read_GEOSldas.py from current working directory")

PROJECT_ROOT = REPO_ROOT / "projects" / "IMS"
NOTEBOOK_OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "outputs_ims_ol_da_validation"
NOTEBOOK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

_cache_tag = f"{DOMAIN}_{YEAR_START}_{YEAR_END}_thr{SCF_THRESHOLD:.2f}".replace(".", "p")
DAILY_COUNTS_PARQUET = NOTEBOOK_OUTPUT_DIR / f"ims_ol_da_daily_counts_{_cache_tag}.parquet"
DAILY_COUNTS_CSV = NOTEBOOK_OUTPUT_DIR / f"ims_ol_da_daily_counts_{_cache_tag}.csv"

PAIR_DAILY_PARQUET = NOTEBOOK_OUTPUT_DIR / f"ims_ol_da_pair_daily_{_cache_tag}.parquet"
PAIR_DAILY_CSV = NOTEBOOK_OUTPUT_DIR / f"ims_ol_da_pair_daily_{_cache_tag}.csv"

COMPARISON_TABLE_PARQUET = NOTEBOOK_OUTPUT_DIR / f"ims_ol_da_comparison_table_{_cache_tag}.parquet"
COMPARISON_TABLE_CSV = NOTEBOOK_OUTPUT_DIR / f"ims_ol_da_comparison_table_{_cache_tag}.csv"

# Repo-local import for tilecoord.
sys.path.insert(0, str(REPO_ROOT / "common/python/io"))
from read_GEOSldas import read_tilecoord  # type: ignore

print(f"REPO_ROOT={REPO_ROOT}")
print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"NOTEBOOK_OUTPUT_DIR={NOTEBOOK_OUTPUT_DIR}")
print(f"DOMAIN={DOMAIN}")
print(f"YEAR_START={YEAR_START}, YEAR_END={YEAR_END}")
print(f"IMS_REGRID_DIR={IMS_REGRID_DIR}")
print(f"IMS_REGRID_TEMPLATE={IMS_REGRID_TEMPLATE}")
print(f"IMS_VAR={IMS_VAR}")
print(f"MODEL_SCF_VAR_CANDIDATES={MODEL_SCF_VAR_CANDIDATES}")
print(f"SCF_THRESHOLD={SCF_THRESHOLD}")
print(f"AUTO_INFER_IMS_CODES={AUTO_INFER_IMS_CODES}")
print(f"IMS_SNOW_CODES={sorted(IMS_SNOW_CODES)}")
print(f"COMPARE_EXP_OL={COMPARE_EXP_OL}, COMPARE_EXP_DA={COMPARE_EXP_DA}")
print(f"Bootstrap: n={N_BOOTSTRAP}, seed={BOOTSTRAP_SEED}, CI=[{CI_LOW}, {CI_HIGH}]")
print(f"DAILY_COUNTS_PARQUET={DAILY_COUNTS_PARQUET}")
print(f"PAIR_DAILY_PARQUET={PAIR_DAILY_PARQUET}")
print(f"COMPARISON_TABLE_PARQUET={COMPARISON_TABLE_PARQUET}")
for k, cfg in EXPERIMENTS.items():
    print(f"{k}: exp_name={cfg['exp_name']}, run_root={cfg['run_root']}")


In [ ]:
# -------------------------
# Helpers
# -------------------------
def season_name(ts: pd.Timestamp) -> str:
    m = int(ts.month)
    if m in (12, 1, 2):
        return "DJF"
    if m in (3, 4, 5):
        return "MAM"
    if m in (6, 7, 8):
        return "JJA"
    return "SON"


def locate_tilecoord_file(run_root: Path, exp_name: str, domain: str, output_dir: Path) -> Path:
    candidates = [
        output_dir / "tilecoord.bin",
        output_dir / f"{exp_name}.ldas_tilecoord.bin",
        run_root / "output" / domain / "rc_out" / f"{exp_name}.ldas_tilecoord.bin",
        run_root / exp_name / "output" / domain / "rc_out" / f"{exp_name}.ldas_tilecoord.bin",
        run_root / f"{exp_name}.ldas_tilecoord.bin",
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError("Could not find tilecoord file. Checked: " + ", ".join([str(p) for p in candidates]))


def locate_daily_cat_file(run_root: Path, exp_name: str, domain: str, day: pd.Timestamp):
    y = f"Y{day.year:04d}"
    m = f"M{day.month:02d}"
    stamp = day.strftime("%Y%m%d")
    fname = f"{exp_name}.tavg24_1d_lnd_Nt.{stamp}_1200z.nc4"
    candidates = [
        run_root / "output" / domain / "cat" / "ens_avg" / y / m / fname,
        run_root / exp_name / "output" / domain / "cat" / "ens_avg" / y / m / fname,
        run_root / "cat" / "ens_avg" / y / m / fname,
    ]
    for p in candidates:
        if p.exists():
            return p
    return None


def choose_representative_tile_per_cell(tc: dict) -> dict:
    i_indg = np.asarray(tc["i_indg"], dtype=np.int64)
    j_indg = np.asarray(tc["j_indg"], dtype=np.int64)
    tile_id = np.asarray(tc["tile_id"], dtype=np.int64)
    frac_cell = np.asarray(tc["frac_cell"], dtype=np.float64)

    frac_sort = np.where(np.isfinite(frac_cell), frac_cell, -np.inf)

    nx = int(i_indg.max()) + 1
    ny = int(j_indg.max()) + 1
    cell_code = j_indg * nx + i_indg

    order = np.lexsort((tile_id, -frac_sort, cell_code))
    code_sorted = cell_code[order]

    first = np.empty(code_sorted.size, dtype=bool)
    first[0] = True
    first[1:] = code_sorted[1:] != code_sorted[:-1]
    rep_idx = order[first]

    return {
        "rep_idx": rep_idx.astype(np.int64),
        "rep_i": i_indg[rep_idx].astype(np.int32),
        "rep_j": j_indg[rep_idx].astype(np.int32),
        "rep_tile_id": tile_id[rep_idx].astype(np.int32),
        "rep_frac_cell": frac_cell[rep_idx].astype(np.float32),
        "rep_lat": np.asarray(tc["com_lat"][rep_idx], dtype=np.float64),
        "rep_lon": np.asarray(tc["com_lon"][rep_idx], dtype=np.float64),
        "rep_elev_m": np.asarray(tc["elev"][rep_idx], dtype=np.float64),
        "nx": np.int32(nx),
        "ny": np.int32(ny),
    }


def find_model_scf_var(ds: xr.Dataset, candidates) -> str:
    for name in candidates:
        if name in ds.variables:
            return name
    raise KeyError(f"None of model SCF variable candidates found: {candidates}")


def read_model_scf_grid_for_rep(nc_path: Path, rep: dict, var_name=None):
    with xr.open_dataset(nc_path, decode_times=False) as ds:
        used_var = var_name if var_name is not None else find_model_scf_var(ds, MODEL_SCF_VAR_CANDIDATES)
        if used_var not in ds.variables:
            raise KeyError(f"Model SCF variable '{used_var}' not found in {nc_path}")
        da = ds[used_var]
        if "time" in da.dims:
            da = da.isel(time=0)
        vals = np.asarray(da.values, dtype=np.float32).reshape(-1)

    rep_idx = rep["rep_idx"]
    if vals.size <= int(rep_idx.max()):
        raise ValueError(
            f"Model var tile length {vals.size} smaller than representative index max {int(rep_idx.max())}"
        )

    sel = np.asarray(vals[rep_idx], dtype=np.float32)
    sel[(sel > 1e14) | (sel < 0.0)] = np.nan

    ny = int(rep["ny"])
    nx = int(rep["nx"])
    grid = np.full((ny, nx), np.nan, dtype=np.float32)
    grid[rep["rep_j"], rep["rep_i"]] = sel

    return grid, used_var


def _decode_time_like(var: xr.DataArray, n: int, year: int):
    if var.ndim != 1 or var.shape[0] != n:
        return None

    vals = np.asarray(var.values)
    if np.issubdtype(vals.dtype, np.datetime64):
        return pd.DatetimeIndex(pd.to_datetime(vals))

    units = str(var.attrs.get("units", ""))
    if "since" in units:
        try:
            base_txt = units.split("since", 1)[1].strip().split()[0]
            base = pd.Timestamp(base_txt)
            return pd.DatetimeIndex(base + pd.to_timedelta(vals.astype(float), unit="D"))
        except Exception:
            return None

    return None


def get_ims_time_dim_and_dates(ds: xr.Dataset, ims_var: str, year: int):
    da = ds[ims_var]
    if da.ndim != 3:
        raise ValueError(f"IMS var {ims_var} must be 3D, found dims={da.dims}")

    time_dim = da.dims[0]
    n_time = da.shape[0]

    if "doy" in ds.variables and ds["doy"].ndim == 1 and ds["doy"].shape[0] == n_time:
        doy = np.asarray(ds["doy"].values, dtype=float)
        base = pd.Timestamp(f"{year}-01-01")
        dates = pd.DatetimeIndex(base + pd.to_timedelta(doy - 1.0, unit="D"))
        return time_dim, dates

    for name in (time_dim, "time", "day_of_year"):
        if name in ds.variables:
            decoded = _decode_time_like(ds[name], n_time, year)
            if decoded is not None:
                return time_dim, decoded

    base = pd.Timestamp(f"{year}-01-01")
    dates = pd.DatetimeIndex(base + pd.to_timedelta(np.arange(n_time), unit="D"))
    return time_dim, dates


def infer_ims_code_sets(unique_codes: set[int]):
    code_set = {int(c) for c in unique_codes if int(c) not in IMS_FILL_VALUES}
    if not code_set:
        raise ValueError("No usable IMS category codes available for inference")

    if code_set.issubset({0, 1}):
        return {1}, {0}
    if 4 in code_set:
        return {4}, {c for c in code_set if c != 4}
    if 1 in code_set:
        return {1}, {c for c in code_set if c != 1}

    raise ValueError(
        "Could not infer IMS snow/no-snow code sets from observed codes: "
        f"{sorted(code_set)}. Set AUTO_INFER_IMS_CODES=False and configure codes manually."
    )


def ims_category_to_binary_scf(cat_grid, snow_codes: set[int], no_snow_codes: set[int], fill_values: set[int]):
    arr = np.asarray(cat_grid, dtype=np.float32)
    finite = np.isfinite(arr)

    arr_i = np.full(arr.shape, -99999, dtype=np.int32)
    arr_i[finite] = np.rint(arr[finite]).astype(np.int32)

    valid_not_fill = finite.copy()
    for fv in fill_values:
        valid_not_fill &= (arr_i != int(fv))

    known_codes = set(snow_codes) | set(no_snow_codes)
    known_mask = np.isin(arr_i, np.array(sorted(known_codes), dtype=np.int32))

    valid_mask = valid_not_fill & known_mask
    unknown_mask = valid_not_fill & (~known_mask)

    obs_scf = np.full(arr.shape, np.nan, dtype=np.float32)
    snow_mask = valid_mask & np.isin(arr_i, np.array(sorted(snow_codes), dtype=np.int32))
    nosnow_mask = valid_mask & np.isin(arr_i, np.array(sorted(no_snow_codes), dtype=np.int32))

    obs_scf[snow_mask] = np.float32(1.0)
    obs_scf[nosnow_mask] = np.float32(0.0)

    return obs_scf, valid_mask, unknown_mask, arr_i


def contingency_counts_binary(obs_scf_binary, mod_scf, threshold=0.5, extra_mask=None):
    obs = np.asarray(obs_scf_binary, dtype=np.float32)
    mod = np.asarray(mod_scf, dtype=np.float32)

    valid = np.isfinite(obs) & np.isfinite(mod)
    if extra_mask is not None:
        valid &= np.asarray(extra_mask, dtype=bool)

    if not np.any(valid):
        return {"A": 0, "B": 0, "C": 0, "D": 0, "N": 0}

    snow_obs = obs >= np.float32(0.5)
    snow_mod = mod >= np.float32(threshold)

    A = int(np.sum(valid & snow_mod & snow_obs))
    B = int(np.sum(valid & snow_mod & (~snow_obs)))
    C = int(np.sum(valid & (~snow_mod) & snow_obs))
    D = int(np.sum(valid & (~snow_mod) & (~snow_obs)))
    N = int(A + B + C + D)

    return {"A": A, "B": B, "C": C, "D": D, "N": N}


def snow_scores_from_counts(A: int, B: int, C: int, D: int):
    N = A + B + C + D
    accuracy = (A + D) / N if N else np.nan

    den_snow_obs = A + C
    hit_rate = A / den_snow_obs if den_snow_obs else np.nan
    miss_rate = C / den_snow_obs if den_snow_obs else np.nan

    den_snow_mod = A + B
    false_alarm_ratio = B / den_snow_mod if den_snow_mod else np.nan

    den_nosnow_obs = B + D
    correct_rejection_rate = D / den_nosnow_obs if den_nosnow_obs else np.nan

    return {
        "accuracy": float(accuracy) if np.isfinite(accuracy) else np.nan,
        "hit_rate": float(hit_rate) if np.isfinite(hit_rate) else np.nan,
        "miss_rate": float(miss_rate) if np.isfinite(miss_rate) else np.nan,
        "false_alarm_ratio": float(false_alarm_ratio) if np.isfinite(false_alarm_ratio) else np.nan,
        "correct_rejection_rate": float(correct_rejection_rate) if np.isfinite(correct_rejection_rate) else np.nan,
    }


METRIC_ORDER = [
    "accuracy", "hit_rate", "miss_rate", "false_alarm_ratio", "correct_rejection_rate",
]


def all_metrics_from_counts(A: int, B: int, C: int, D: int):
    return snow_scores_from_counts(A, B, C, D)


def _pct_finite(x, p):
    a = np.asarray(x, dtype=np.float64)
    a = a[np.isfinite(a)]
    if a.size == 0:
        return np.nan
    return float(np.percentile(a, p))


def bootstrap_compare_table_from_pair_days(days_df: pd.DataFrame, exp_ol: str, exp_da: str, n_boot=2000, seed=42):
    """
    Paired bootstrap over day blocks.

    For each bootstrap replicate we resample day indices with replacement, and for each
    selected day use both OL and DA counts from the same day (same collocated samples).
    """
    req_cols = [
        f"A_{exp_ol}", f"B_{exp_ol}", f"C_{exp_ol}", f"D_{exp_ol}",
        f"A_{exp_da}", f"B_{exp_da}", f"C_{exp_da}", f"D_{exp_da}",
    ]
    missing = [c for c in req_cols if c not in days_df.columns]
    if missing:
        raise KeyError(f"Missing required paired columns: {missing}")

    n_days = len(days_df)
    if n_days == 0:
        return pd.DataFrame()

    ol_counts = days_df[[f"A_{exp_ol}", f"B_{exp_ol}", f"C_{exp_ol}", f"D_{exp_ol}"]].to_numpy(dtype=np.int64)
    da_counts = days_df[[f"A_{exp_da}", f"B_{exp_da}", f"C_{exp_da}", f"D_{exp_da}"]].to_numpy(dtype=np.int64)

    # Point estimates from full set.
    A_ol, B_ol, C_ol, D_ol = np.sum(ol_counts, axis=0)
    A_da, B_da, C_da, D_da = np.sum(da_counts, axis=0)

    point_ol = all_metrics_from_counts(int(A_ol), int(B_ol), int(C_ol), int(D_ol))
    point_da = all_metrics_from_counts(int(A_da), int(B_da), int(C_da), int(D_da))

    rng = np.random.default_rng(int(seed))

    boot_ol = {m: [] for m in METRIC_ORDER}
    boot_da = {m: [] for m in METRIC_ORDER}
    boot_delta = {m: [] for m in METRIC_ORDER}

    for _ in range(int(n_boot)):
        idx = rng.integers(0, n_days, size=n_days)
        b_ol = np.sum(ol_counts[idx, :], axis=0)
        b_da = np.sum(da_counts[idx, :], axis=0)

        m_ol = all_metrics_from_counts(int(b_ol[0]), int(b_ol[1]), int(b_ol[2]), int(b_ol[3]))
        m_da = all_metrics_from_counts(int(b_da[0]), int(b_da[1]), int(b_da[2]), int(b_da[3]))

        for m in METRIC_ORDER:
            v_ol = m_ol[m]
            v_da = m_da[m]
            boot_ol[m].append(v_ol)
            boot_da[m].append(v_da)
            if np.isfinite(v_ol) and np.isfinite(v_da):
                boot_delta[m].append(v_da - v_ol)
            else:
                boot_delta[m].append(np.nan)

    rows = []
    for m in METRIC_ORDER:
        ol = point_ol[m]
        da = point_da[m]
        delta = (da - ol) if (np.isfinite(ol) and np.isfinite(da)) else np.nan

        rows.append(
            {
                "metric": m,
                "ol": float(ol) if np.isfinite(ol) else np.nan,
                "ol_ci_lo": _pct_finite(boot_ol[m], CI_LOW),
                "ol_ci_hi": _pct_finite(boot_ol[m], CI_HIGH),
                "da": float(da) if np.isfinite(da) else np.nan,
                "da_ci_lo": _pct_finite(boot_da[m], CI_LOW),
                "da_ci_hi": _pct_finite(boot_da[m], CI_HIGH),
                "delta_da_minus_ol": float(delta) if np.isfinite(delta) else np.nan,
                "delta_ci_lo": _pct_finite(boot_delta[m], CI_LOW),
                "delta_ci_hi": _pct_finite(boot_delta[m], CI_HIGH),
                "n_days": int(n_days),
                "A_ol": int(A_ol),
                "B_ol": int(B_ol),
                "C_ol": int(C_ol),
                "D_ol": int(D_ol),
                "N_ol": int(A_ol + B_ol + C_ol + D_ol),
                "A_da": int(A_da),
                "B_da": int(B_da),
                "C_da": int(C_da),
                "D_da": int(D_da),
                "N_da": int(A_da + B_da + C_da + D_da),
            }
        )

    return pd.DataFrame(rows)


def daily_cache_valid(df: pd.DataFrame) -> bool:
    required = {
        "date", "year", "doy", "season", "experiment",
        "A", "B", "C", "D", "N_valid",
        "accuracy", "hit_rate", "miss_rate", "false_alarm_ratio", "correct_rejection_rate",
        "model_var", "model_file_found", "model_read_ok",
        "N_land_mask", "N_ims_obs_valid", "N_ims_unknown_codes",
        "obs_scf_mean", "mod_scf_mean",
        "paired_common_mask_used", "N_common_mask",
    }
    if not required.issubset(set(df.columns)):
        return False
    if df.empty:
        return False
    y0 = int(df["year"].min())
    y1 = int(df["year"].max())
    return (y0 <= YEAR_START) and (y1 >= YEAR_END)


In [ ]:
# -------------------------
# Build or load daily intermediate cache
# -------------------------
daily_df = None

if USE_DAILY_CACHE and DAILY_COUNTS_PARQUET.exists():
    print(f"Found daily cache: {DAILY_COUNTS_PARQUET}")
    tmp = pd.read_parquet(DAILY_COUNTS_PARQUET)
    tmp["date"] = pd.to_datetime(tmp["date"])
    if daily_cache_valid(tmp):
        daily_df = tmp.copy()
        print("Using existing daily cache (coverage/columns look valid).")
    else:
        print("Existing daily cache failed validation; re-running extraction.")

if daily_df is None:
    print("Running extraction from IMS regridded + model daily cat files...")

    exp_rep = {}
    exp_model_var = {}
    target_shape = None

    exp_keys = list(EXPERIMENTS.keys())

    for exp_key, cfg in EXPERIMENTS.items():
        run_root = Path(cfg["run_root"])
        exp_name = str(cfg["exp_name"])

        tilecoord_path = locate_tilecoord_file(run_root, exp_name, DOMAIN, NOTEBOOK_OUTPUT_DIR)
        tc = read_tilecoord(str(tilecoord_path))
        rep = choose_representative_tile_per_cell(tc)

        ny = int(rep["ny"])
        nx = int(rep["nx"])
        print(f"{exp_key}: tilecoord={tilecoord_path}")
        print(f"{exp_key}: rep cells={rep['rep_idx'].size}, grid shape=(ny={ny}, nx={nx})")

        if target_shape is None:
            target_shape = (ny, nx)
        elif target_shape != (ny, nx):
            raise ValueError(f"Representative grid mismatch across experiments: {target_shape} vs {(ny, nx)}")

        # Detect model SCF var from first available daily cat file in requested window.
        detected_var = None
        for day in pd.date_range(f"{YEAR_START}-01-01", f"{YEAR_END}-12-31", freq="D"):
            fp = locate_daily_cat_file(run_root, exp_name, DOMAIN, pd.Timestamp(day))
            if fp is None:
                continue
            with xr.open_dataset(fp, decode_times=False) as ds_sample:
                detected_var = find_model_scf_var(ds_sample, MODEL_SCF_VAR_CANDIDATES)
            break

        if detected_var is None:
            warnings.warn(
                f"{exp_key}: could not detect model SCF var from available files. "
                "Will try to auto-detect per file during extraction."
            )

        exp_rep[exp_key] = rep
        exp_model_var[exp_key] = detected_var

    if target_shape is None:
        raise RuntimeError("No experiment representative shape available")

    records = []

    all_days = pd.date_range(f"{YEAR_START}-01-01", f"{YEAR_END}-12-31", freq="D")
    all_days_set = {pd.Timestamp(d).date() for d in all_days}

    for year in range(YEAR_START, YEAR_END + 1):
        ims_path = IMS_REGRID_DIR / IMS_REGRID_TEMPLATE.format(year=year)
        if not ims_path.exists():
            warnings.warn(f"Missing IMS regridded file for year {year}: {ims_path}")
            continue

        print(f"Year {year}: reading IMS {ims_path}")

        with xr.open_dataset(ims_path, decode_times=False) as ds_ims:
            if IMS_VAR not in ds_ims.variables:
                warnings.warn(f"{ims_path} missing {IMS_VAR}; skipping year")
                continue

            time_dim, ims_dates = get_ims_time_dim_and_dates(ds_ims, IMS_VAR, year)
            n_time = ds_ims[IMS_VAR].shape[0]
            ims_ny = int(ds_ims[IMS_VAR].shape[1])
            ims_nx = int(ds_ims[IMS_VAR].shape[2])

            if target_shape != (ims_ny, ims_nx):
                raise ValueError(
                    f"IMS shape {(ims_ny, ims_nx)} does not match representative shape {target_shape}. "
                    "Use matching tilecoord/regrid setup."
                )

            if "land_mask" in ds_ims.variables:
                land_mask = np.asarray(ds_ims["land_mask"].values, dtype=np.int8) == 1
            else:
                land_mask = np.ones((ims_ny, ims_nx), dtype=bool)

            if "within_max_distance" in ds_ims.variables:
                within_mask = np.asarray(ds_ims["within_max_distance"].values, dtype=np.int8) == 1
            else:
                within_mask = np.ones((ims_ny, ims_nx), dtype=bool)

            static_mask = land_mask & within_mask

            if AUTO_INFER_IMS_CODES:
                seen_codes = set()
                n_sample = min(5, n_time)
                for si in range(n_sample):
                    sample = np.asarray(ds_ims[IMS_VAR].isel({time_dim: si}).values, dtype=np.float32)
                    finite = np.isfinite(sample)
                    if np.any(finite):
                        sample_codes = np.unique(np.rint(sample[finite]).astype(np.int32))
                        for c in sample_codes:
                            ci = int(c)
                            if ci not in IMS_FILL_VALUES:
                                seen_codes.add(ci)

                if seen_codes:
                    snow_codes, no_snow_codes = infer_ims_code_sets(seen_codes)
                else:
                    snow_codes = set(IMS_SNOW_CODES)
                    no_snow_codes = set(IMS_NO_SNOW_CODES)
            else:
                snow_codes = set(IMS_SNOW_CODES)
                no_snow_codes = set(IMS_NO_SNOW_CODES)

            print(
                f"Year {year}: IMS snow_codes={sorted(snow_codes)}, "
                f"no_snow_codes={sorted(no_snow_codes)}"
            )

            for ti in range(n_time):
                day = pd.Timestamp(ims_dates[ti])
                day_date = day.date()
                if day_date not in all_days_set:
                    continue

                ims_cat = np.asarray(ds_ims[IMS_VAR].isel({time_dim: ti}).values, dtype=np.float32)
                obs_bin, obs_valid_mask, unknown_mask, _ = ims_category_to_binary_scf(
                    ims_cat,
                    snow_codes=snow_codes,
                    no_snow_codes=no_snow_codes,
                    fill_values=IMS_FILL_VALUES,
                )

                base_valid_mask = static_mask & obs_valid_mask
                n_land_mask = int(np.sum(static_mask))
                n_obs_valid = int(np.sum(base_valid_mask))
                n_unknown = int(np.sum(static_mask & unknown_mask))

                if (ti + 1) % 30 == 0 or (ti + 1) == n_time:
                    print(f"  IMS day {ti + 1}/{n_time}: {day.strftime('%Y-%m-%d')}")

                # First pass: read model grids for all experiments for this day.
                rec_by_exp = {}
                grid_by_exp = {}

                for exp_key, cfg in EXPERIMENTS.items():
                    run_root = Path(cfg["run_root"])
                    exp_name = str(cfg["exp_name"])
                    rep = exp_rep[exp_key]

                    rec = {
                        "date": day.normalize(),
                        "year": int(day.year),
                        "doy": int(day.strftime("%j")),
                        "season": season_name(day),
                        "experiment": exp_key,
                        "exp_name": exp_name,
                        "ims_file": str(ims_path),
                        "model_file": None,
                        "model_file_found": 0,
                        "model_read_ok": 0,
                        "model_var": exp_model_var.get(exp_key),
                        "A": 0,
                        "B": 0,
                        "C": 0,
                        "D": 0,
                        "N_valid": 0,
                        "accuracy": np.nan,
                        "hit_rate": np.nan,
                        "miss_rate": np.nan,
                        "false_alarm_ratio": np.nan,
                        "correct_rejection_rate": np.nan,
                        "N_land_mask": n_land_mask,
                        "N_ims_obs_valid": n_obs_valid,
                        "N_ims_unknown_codes": n_unknown,
                        "obs_scf_mean": np.nan,
                        "mod_scf_mean": np.nan,
                        "paired_common_mask_used": 0,
                        "N_common_mask": 0,
                        "mask_type": "none",
                    }

                    model_path = locate_daily_cat_file(run_root, exp_name, DOMAIN, day)
                    if model_path is None:
                        rec_by_exp[exp_key] = rec
                        continue

                    rec["model_file"] = str(model_path)
                    rec["model_file_found"] = 1

                    try:
                        mod_grid, used_var = read_model_scf_grid_for_rep(
                            model_path,
                            rep=rep,
                            var_name=exp_model_var.get(exp_key),
                        )
                        rec["model_var"] = used_var
                        rec["model_read_ok"] = 1
                        if exp_model_var.get(exp_key) is None:
                            exp_model_var[exp_key] = used_var
                        grid_by_exp[exp_key] = mod_grid
                    except Exception as exc:
                        warnings.warn(f"{exp_key} {day.strftime('%Y-%m-%d')} read failed: {exc}")

                    rec_by_exp[exp_key] = rec

                # Second pass: compute counts.
                both_ok = all(rec_by_exp[e]["model_read_ok"] == 1 for e in exp_keys)

                if both_ok:
                    common_mask = base_valid_mask.copy()
                    for e in exp_keys:
                        common_mask &= np.isfinite(grid_by_exp[e])

                    n_common = int(np.sum(common_mask))

                    for e in exp_keys:
                        counts = contingency_counts_binary(
                            obs_bin,
                            grid_by_exp[e],
                            threshold=SCF_THRESHOLD,
                            extra_mask=common_mask,
                        )
                        m = all_metrics_from_counts(counts["A"], counts["B"], counts["C"], counts["D"])

                        rec = rec_by_exp[e]
                        rec["A"] = int(counts["A"])
                        rec["B"] = int(counts["B"])
                        rec["C"] = int(counts["C"])
                        rec["D"] = int(counts["D"])
                        rec["N_valid"] = int(counts["N"])
                        rec["accuracy"] = m["accuracy"]
                        rec["hit_rate"] = m["hit_rate"]
                        rec["miss_rate"] = m["miss_rate"]
                        rec["false_alarm_ratio"] = m["false_alarm_ratio"]
                        rec["correct_rejection_rate"] = m["correct_rejection_rate"]
                        rec["paired_common_mask_used"] = 1
                        rec["N_common_mask"] = n_common
                        rec["mask_type"] = "common_pair"

                        valid_mean_mask = common_mask & np.isfinite(grid_by_exp[e])
                        if np.any(valid_mean_mask):
                            rec["obs_scf_mean"] = float(np.nanmean(obs_bin[valid_mean_mask]))
                            rec["mod_scf_mean"] = float(np.nanmean(grid_by_exp[e][valid_mean_mask]))

                        rec_by_exp[e] = rec
                else:
                    # Optional individual counts (diagnostics only; not used in fair OL-vs-DA table).
                    for e in exp_keys:
                        rec = rec_by_exp[e]
                        if rec["model_read_ok"] == 1:
                            indiv_mask = base_valid_mask & np.isfinite(grid_by_exp[e])
                            counts = contingency_counts_binary(
                                obs_bin,
                                grid_by_exp[e],
                                threshold=SCF_THRESHOLD,
                                extra_mask=indiv_mask,
                            )
                            m = all_metrics_from_counts(counts["A"], counts["B"], counts["C"], counts["D"])

                            rec["A"] = int(counts["A"])
                            rec["B"] = int(counts["B"])
                            rec["C"] = int(counts["C"])
                            rec["D"] = int(counts["D"])
                            rec["N_valid"] = int(counts["N"])
                            rec["accuracy"] = m["accuracy"]
                            rec["hit_rate"] = m["hit_rate"]
                            rec["miss_rate"] = m["miss_rate"]
                            rec["false_alarm_ratio"] = m["false_alarm_ratio"]
                            rec["correct_rejection_rate"] = m["correct_rejection_rate"]
                            rec["mask_type"] = "individual"

                            if np.any(indiv_mask):
                                rec["obs_scf_mean"] = float(np.nanmean(obs_bin[indiv_mask]))
                                rec["mod_scf_mean"] = float(np.nanmean(grid_by_exp[e][indiv_mask]))

                            rec_by_exp[e] = rec

                for e in exp_keys:
                    records.append(rec_by_exp[e])

    daily_df = pd.DataFrame.from_records(records)
    if daily_df.empty:
        raise RuntimeError("No daily records generated. Check input paths and year range.")

    daily_df["date"] = pd.to_datetime(daily_df["date"])
    daily_df = daily_df.sort_values(["date", "experiment"]).reset_index(drop=True)

    if WRITE_DAILY_CACHE:
        daily_df.to_parquet(DAILY_COUNTS_PARQUET, index=False)
        daily_df.to_csv(DAILY_COUNTS_CSV, index=False)
        print(f"Wrote daily cache parquet: {DAILY_COUNTS_PARQUET}")
        print(f"Wrote daily cache csv: {DAILY_COUNTS_CSV}")

print("Daily dataframe shape:", daily_df.shape)
print("Date span:", daily_df["date"].min(), "to", daily_df["date"].max())
print("Paired common-mask rows:", int((daily_df["paired_common_mask_used"] == 1).sum()))
display(daily_df.head(20))


In [ ]:
# -------------------------
# Build paired OL-vs-DA day table, bootstrap CIs, and save output table
# -------------------------
if COMPARE_EXP_OL not in EXPERIMENTS or COMPARE_EXP_DA not in EXPERIMENTS:
    raise ValueError(
        f"COMPARE_EXP_OL/DA must be keys in EXPERIMENTS. Got {COMPARE_EXP_OL}, {COMPARE_EXP_DA}."
    )

paired_rows = daily_df[
    (daily_df["paired_common_mask_used"] == 1)
    & (daily_df["model_read_ok"] == 1)
    & (daily_df["N_valid"] > 0)
].copy()

if paired_rows.empty:
    raise RuntimeError("No paired common-mask rows available for OL-vs-DA comparison")

pair_daily = (
    paired_rows.pivot_table(
        index=["date", "year", "doy", "season"],
        columns="experiment",
        values=[
            "A", "B", "C", "D", "N_valid",
            "obs_scf_mean", "mod_scf_mean",
            "model_file_found", "model_read_ok", "paired_common_mask_used",
        ],
        aggfunc="first",
    )
)

pair_daily.columns = [f"{v}_{e}" for v, e in pair_daily.columns]
pair_daily = pair_daily.reset_index()

# Keep days where both comparison experiments are present and valid.
need_cols = [
    f"A_{COMPARE_EXP_OL}", f"B_{COMPARE_EXP_OL}", f"C_{COMPARE_EXP_OL}", f"D_{COMPARE_EXP_OL}", f"N_valid_{COMPARE_EXP_OL}",
    f"A_{COMPARE_EXP_DA}", f"B_{COMPARE_EXP_DA}", f"C_{COMPARE_EXP_DA}", f"D_{COMPARE_EXP_DA}", f"N_valid_{COMPARE_EXP_DA}",
]
for c in need_cols:
    if c not in pair_daily.columns:
        pair_daily[c] = np.nan

valid_pair_mask = (
    pair_daily[f"N_valid_{COMPARE_EXP_OL}"].fillna(0) > 0
) & (
    pair_daily[f"N_valid_{COMPARE_EXP_DA}"].fillna(0) > 0
)

pair_daily = pair_daily[valid_pair_mask].copy()
if pair_daily.empty:
    raise RuntimeError("No paired valid days left after OL/DA pivot/filter")

pair_daily = pair_daily.sort_values(["date"]).reset_index(drop=True)
pair_daily.to_parquet(PAIR_DAILY_PARQUET, index=False)
pair_daily.to_csv(PAIR_DAILY_CSV, index=False)
print(f"Wrote paired daily parquet: {PAIR_DAILY_PARQUET}")
print(f"Wrote paired daily csv: {PAIR_DAILY_CSV}")
print("Paired daily shape:", pair_daily.shape)


# Build comparison table across requested scopes.
comparison_parts = []


def add_scope(df_scope: pd.DataFrame, scope_name: str, year_val=np.nan, season_val=np.nan, seed_offset=0):
    if df_scope.empty:
        return
    tbl = bootstrap_compare_table_from_pair_days(
        df_scope,
        exp_ol=COMPARE_EXP_OL,
        exp_da=COMPARE_EXP_DA,
        n_boot=N_BOOTSTRAP,
        seed=int(BOOTSTRAP_SEED + seed_offset),
    )
    if tbl.empty:
        return
    tbl["scope"] = scope_name
    tbl["year"] = year_val
    tbl["season"] = season_val
    comparison_parts.append(tbl)


# All period.
add_scope(pair_daily, "ALL_PERIOD", seed_offset=0)

# Season across all years.
for s in SEASON_ORDER:
    add_scope(pair_daily[pair_daily["season"] == s], "SEASON_ALL_YEARS", season_val=s, seed_offset=10 + SEASON_ORDER.index(s))

# Per year.
for y in sorted(pair_daily["year"].unique()):
    y = int(y)
    add_scope(pair_daily[pair_daily["year"] == y], "YEAR", year_val=y, seed_offset=1000 + y)

# Per year-season.
for y in sorted(pair_daily["year"].unique()):
    y = int(y)
    for s in SEASON_ORDER:
        sub = pair_daily[(pair_daily["year"] == y) & (pair_daily["season"] == s)]
        if sub.empty:
            continue
        add_scope(sub, "YEAR_SEASON", year_val=y, season_val=s, seed_offset=2000 + y * 10 + SEASON_ORDER.index(s))


if not comparison_parts:
    raise RuntimeError("No comparison rows produced")

comparison_df = pd.concat(comparison_parts, ignore_index=True, sort=False)
comparison_df = comparison_df[
    [
        "scope", "year", "season", "metric",
        "ol", "ol_ci_lo", "ol_ci_hi",
        "da", "da_ci_lo", "da_ci_hi",
        "delta_da_minus_ol", "delta_ci_lo", "delta_ci_hi",
        "n_days",
        "A_ol", "B_ol", "C_ol", "D_ol", "N_ol",
        "A_da", "B_da", "C_da", "D_da", "N_da",
    ]
].copy()

comparison_df.to_parquet(COMPARISON_TABLE_PARQUET, index=False)
comparison_df.to_csv(COMPARISON_TABLE_CSV, index=False)

print(f"Wrote comparison table parquet: {COMPARISON_TABLE_PARQUET}")
print(f"Wrote comparison table csv: {COMPARISON_TABLE_CSV}")
print("Comparison table shape:", comparison_df.shape)

print("\nAll-period OL vs DA table (with 95% bootstrap CI):")
all_period_tbl = comparison_df[comparison_df["scope"] == "ALL_PERIOD"].copy()
all_period_tbl = all_period_tbl.set_index("metric").reindex(METRIC_ORDER).reset_index()
display(all_period_tbl)

print("\nSeason-all-years OL vs DA table:")
season_tbl = comparison_df[comparison_df["scope"] == "SEASON_ALL_YEARS"].copy()
if not season_tbl.empty:
    season_tbl["season"] = pd.Categorical(season_tbl["season"], categories=SEASON_ORDER, ordered=True)
season_tbl = season_tbl.sort_values(["season", "metric"]).reset_index(drop=True)
display(season_tbl)


## Notes

- This notebook compares **IMS already regridded to M36 EASE** with model daily cat output.
- Fair OL vs DA comparison uses the **same collocated daily samples** via a per-day common mask:
  - `valid = finite(obs) & finite(F_OL) & finite(F_DA) & static_valid`
- Bootstrapping is implemented as a **paired day-block bootstrap**:
  - sample day indices with replacement
  - aggregate `A/B/C/D` for both OL and DA on the same sampled days
  - compute 95% CI from percentile `[2.5, 97.5]`
- Output comparison table includes contingency metrics only
  (`accuracy`, `hit_rate`, `miss_rate`, `false_alarm_ratio`, `correct_rejection_rate`)
  with `OL`, `DA`, and `Δ = DA - OL`, each with CI (`*_ci_lo`, `*_ci_hi`).
- Intermediate caches:
  - `ims_ol_da_daily_counts_*.parquet` (daily extraction records)
  - `ims_ol_da_pair_daily_*.parquet` (paired OL/DA days)
  - `ims_ol_da_comparison_table_*.parquet` (final table)
- If IMS category semantics differ by product/version, set:
  - `AUTO_INFER_IMS_CODES=False`
  - `IMS_SNOW_CODES` and `IMS_NO_SNOW_CODES` explicitly.
